In [1]:
import pandas as pd
import geopandas as gpd
from pynhd import WaterData, NHD

In [2]:
huc2_id = "12"
wd_huc2 = WaterData("wbd02")
wd_huc8 = WaterData("wbd08")
nhd_mr = NHD("flowline_mr")
wd_nhd = WaterData("nhdflowline_network")

huc2 = wd_huc2.byid("huc2", ["12"])
huc8s = wd_huc8.byfilter(f"huc8 LIKE '{huc2_id}%'")

flowline_chunks = []

for row in huc8s.itertuples(index=False):
    huc8_geom_bbox = row.geometry.bounds
    xmin, ymin, xmax, ymax = row.geometry.bounds
    huc8_id = row.huc8

    try:
        lines = wd_nhd.bybox((xmin, ymin, xmax, ymax), box_crs=huc8s.crs)
        # lines = nhd_mr.bygeom(huc8_geom_bbox, geo_crs=huc8s.crs)
        if not lines.empty:
            flowline_chunks.append(lines)
            # print(huc8_id, lines.shape, len(flowline_chunks))
    except Exception:
        print(huc8_id, 'EMPTY', len(flowline_chunks))
        continue

In [3]:
if flowline_chunks:
    all_flowlines = pd.concat(flowline_chunks, ignore_index=True)

    all_flowlines = gpd.GeoDataFrame(all_flowlines, geometry="geometry", crs=flowline_chunks[0].crs)
    
    if all_flowlines.crs != huc2.crs:
        all_flowlines = all_flowlines.to_crs(huc2.crs)

    huc2_flowlines = all_flowlines.clip(huc2.geometry)

    if "nhdplusid" in huc2_flowlines.columns:
        huc2_flowlines = huc2_flowlines.drop_duplicates(subset=["nhdplusid"])
    elif "comid" in huc2_flowlines.columns:
        huc2_flowlines = huc2_flowlines.drop_duplicates(subset=['comid'])
else:
    print('No flowlines fetched')

In [8]:
huc2_flowlines = huc2_flowlines[~(huc2_flowlines['ftype'] == 'Coastline')]
huc2_flowlines.shape

(67311, 138)

In [9]:
import os
os.environ['GDAL_DATA'] = r"C:\Users\vcq32\AppData\Local\miniforge3\Library\share\gdal"
fn = r"C:\Users\vcq32\OneDrive - Texas State University\Projects\EPA_ECHO_Effluent\data\nhdmr_flowline.gpkg"
huc2_flowlines.to_file(fn)

In [12]:
wd_catchment = WaterData("catchmentsp")
comid_list = huc2_flowlines['comid'].unique().tolist()
catchments = wd_catchment.byid("featureid", comid_list)

print(type(catchments), catchments.shape)

: 

In [7]:
huc2_flowlines['ftype'].unique()

<ArrowStringArray>
[    'CanalDitch',    'StreamRiver', 'ArtificialPath',      'Connector',
      'Coastline',       'Pipeline']
Length: 6, dtype: str

In [ ]:
m = huc2_flowlines.explore()
m